# RunPod: install and start ComfyUI
Run all cells. This notebook pins ComfyUI to the commit used by the delivered workflow and never stores API keys.

In [ ]:
from pathlib import Path
import os, platform, shutil, subprocess, sys, time

WORKSPACE = Path(os.environ.get("RUNPOD_VOLUME_PATH", "/workspace"))
COMFYUI_DIR = WORKSPACE / "ComfyUI"
COMFYUI_COMMIT = "b78cec879b9460d5cb25228a83a942fb78d2cd24"
PORT = 8188

print({"python": sys.version, "platform": platform.platform(), "workspace": str(WORKSPACE)})
print(subprocess.run(["nvidia-smi"], text=True, capture_output=True).stdout)
WORKSPACE.mkdir(parents=True, exist_ok=True)

In [ ]:
def run(command):
    print("+", " ".join(map(str, command)))
    subprocess.run(list(map(str, command)), check=True)

if not (COMFYUI_DIR / ".git").exists():
    run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI.git", COMFYUI_DIR])
run(["git", "-C", COMFYUI_DIR, "fetch", "origin", COMFYUI_COMMIT, "--depth", "1"])
run(["git", "-C", COMFYUI_DIR, "checkout", "--detach", COMFYUI_COMMIT])
run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"])
run([sys.executable, "-m", "pip", "install", "-r", COMFYUI_DIR / "requirements.txt"])
actual = subprocess.check_output(["git", "-C", COMFYUI_DIR, "rev-parse", "HEAD"], text=True).strip()
assert actual == COMFYUI_COMMIT, (actual, COMFYUI_COMMIT)
print("ComfyUI ready at", actual)

In [ ]:
log_path = WORKSPACE / "comfyui.log"
log_handle = log_path.open("a", encoding="utf-8")
command = [sys.executable, "main.py", "--listen", "0.0.0.0", "--port", str(PORT)]
process = subprocess.Popen(command, cwd=COMFYUI_DIR, stdout=log_handle, stderr=subprocess.STDOUT)
time.sleep(8)
if process.poll() is not None:
    log_handle.flush()
    raise RuntimeError(f"ComfyUI exited early. Inspect {log_path}")
print(f"ComfyUI started (pid={process.pid}, port={PORT}). Open RunPod HTTP Service port {PORT}.")
print("Log:", log_path)